$$
\newcommand{\argmax}{arg\,max}
\newcommand{\argmin}{arg\,min}
\newcommand{\col}{col}
\newcommand{\Span}{span}
\newcommand{\epsilon}{\varepsilon}
\newcommand{\EE}{\mathbb{E}}
\newcommand{\PP}{\mathbb{P}}
\newcommand{\RR}{\mathbb{R}}
\newcommand{\NN}{\mathbb{N}}
\newcommand{\ZZ}{\mathbb{Z}}
\newcommand{\aA}{\mathcal{A}}
\newcommand{\bB}{\mathcal{B}}
\newcommand{\cC}{\mathcal{C}}
\newcommand{\dD}{\mathcal{D}}
\newcommand{\eE}{\mathcal{E}}
\newcommand{\fF}{\mathcal{F}}
\newcommand{\gG}{\mathcal{G}}
\newcommand{\hH}{\mathcal{H}}
$$


<a id='policy-regimes'></a>

<a id='index-0'></a>

# Policy Regimes and the Lucas Critique

## Overview

Lucas [[1976](https://python-advanced.quantecon.org/zreferences.html#id336)] proposed to evaluate a policy rule by computing outcomes in an economy whose private agents understand the rule and expect it to persist.

McKay and Wolf [[2023](https://python-advanced.quantecon.org/zreferences.html#id335)] compute such outcomes from impulse responses to policy shocks; [Policy Counterfactuals from Impulse Responses](https://python-advanced.quantecon.org/policy_counterfactuals.html) applies their method to a market for a single good.

They describe their thought experiment, in which a monetary policy rule links the interest rate $ i $ to inflation $ \pi $:

> Note that this thought experiment supposes that the private sector perfectly understands the change in rule: the new relationship between $ i $ and $ \pi $ holds for all $ t \geq 0 $.


Their counterfactual is a scenario: a rule announced at date 0, believed by everyone, and kept forever.

Sargent and Wallace [[1976](https://python-advanced.quantecon.org/zreferences.html#id343)] questioned the internal consistency of that experiment:

> … new rules are not adopted in a vacuum. Something would cause the change – a change in administrations, new appointments, and so on. Moreover, if rational agents live in a world in which rules can be and are changed, their behavior should take into account such possibilities and should depend on the process generating the rule changes.


Sims [[1982](https://python-advanced.quantecon.org/zreferences.html#id344)] described most policy changes as realizations of a policy process that private agents already understand.

Sims and Zha [[2006](https://python-advanced.quantecon.org/zreferences.html#id345)] linked vector autoregressions with regime-switching coefficients to the Lucas critique:

> … once we recognize that changes in policy must in principle themselves be modeled as stochastic, Lucas’s argument can be seen as a claim that a certain sort of nonlinearity is important. Even if the public believes that policy is time-varying and tries to adjust its expectation formation accordingly, its behavior could be well approximated as linear and non-time-varying.


This lecture builds a model in which policy rules switch according to a Markov chain that private agents know.

The model is the market of [Dynamic Supply and Demand Curves](https://python-advanced.quantecon.org/supply_demand_var.html) and [Policy Counterfactuals from Impulse Responses](https://python-advanced.quantecon.org/policy_counterfactuals.html), with a tax rule whose coefficient depends on the state of the chain.

The machinery is the Markov jump linear quadratic dynamic programming of [Markov Jump Linear Quadratic Dynamic Programming](https://python-advanced.quantecon.org/markov_jump_lq.html).

The lecture

- constructs a planning problem whose solution is the market equilibrium under switching policy regimes,  
- computes decision rules that depend on the current regime and on the probabilities of future regimes,  
- distinguishes a switch between regimes of a known process from a change in the process itself,  
- fits vector autoregressions to data generated by switching regimes,  
- compares a permanent, credible rule with a reform that agents expect may be reversed,  
- tests McKay and Wolf’s construction when the historical rule switched regimes,  
- extends the construction, following Caravello *et al.* [[2026](https://python-advanced.quantecon.org/zreferences.html#id351)], to a counterfactual rule that may be reversed, and  
- shows, following Manuel and Wolf [[2026](https://python-advanced.quantecon.org/zreferences.html#id352)], how regime switches in the data can take the place of policy shocks in measuring the effects of policy.  


Three questions about policy rules call for three approaches.

[](https://python-advanced.quantecon.org/policy_counterfactuals.html)[](https://python-advanced.quantecon.org/calvo_abreu.html)[](https://python-advanced.quantecon.org/dovis_accounting_mf.html)|question|approach|source of credibility|
|:-------------------------------:|:-------------------------------:|:-------------------------------:|
|what would happen if a rule were announced at date 0 and believed forever?|scenario analysis, as in Policy Counterfactuals from Impulse Responses|assumed|
|which rules would a government choose and be able to sustain?|dynamic programming squared, as in Sustainable Plans for a Calvo Model and Accounting for Monetary and Fiscal Policy|the government’s incentives|
|what happens when rule changes are realizations of a process that agents know?|Markov-switching regimes, this lecture|transition probabilities|
We use the following imports.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## A market with policy regimes

We use the parameters of [Dynamic Supply and Demand Curves](https://python-advanced.quantecon.org/supply_demand_var.html), except that we set the demander’s adjustment cost to zero, $ g_d = 0 $.

With $ g_d = 0 $ the market equilibrium under switching regimes solves a Markov jump linear quadratic planning problem.

The supplier chooses $ q_t $ to maximize

$$
\mathbb{E}_0 \sum_{t=0}^\infty \beta^t \Big\{ (p_t - \tau_t) q_t - \tfrac{h_s}{2} q_t^2 - \tfrac{g_s}{2} (q_t - q_{t-1})^2 - s_t q_t \Big\} ,
$$

and the demander’s inverse demand curve is

$$
p_t = - d_t - h_d q_t .
$$

The cost and demand shocks are $ s_t = B_s(L) w_{st} $ and $ d_t = B_d(L) w_{dt} $, as in [Dynamic Supply and Demand Curves](https://python-advanced.quantecon.org/supply_demand_var.html), where $ w_{st} $ and $ w_{dt} $ are independent with standard deviations $ \sigma_s $ and $ \sigma_d $.

A Markov state $ s^p_t \in \{0, \ldots, m-1\} $ with transition matrix $ \Pi $ selects the policy regime, with $ \Pi_{ij} = \Pr(s^p_{t+1} = j \mid s^p_t = i) $; we number regimes from 0, as the code does.

The government sets the tax by the rule


<a id='equation-pr-rule'></a>
$$
\tau_t = - \phi_{s^p_t} \, p_t + \nu_t , \tag{31.1}
$$

where $ \phi_i $ is the feedback coefficient in regime $ i $ and $ \nu_t $ collects contemporaneous and news shocks to the rule, as in [Policy Counterfactuals from Impulse Responses](https://python-advanced.quantecon.org/policy_counterfactuals.html).

Private agents observe $ s^p_t $ and know $ \Pi $.

### An equivalent planning problem

Suppliers receive $ p_t - \tau_t = (1 + \phi_{s^p_t}) p_t - \nu_t $.

The supplier’s Euler equation and the demand curve imply


<a id='equation-pr-euler'></a>
$$
h_s q_t + g_s \big[ (q_t - q_{t-1}) - \beta \, \mathbb{E}_t (q_{t+1} - q_t) \big] + s_t + \nu_t
+ (1 + \phi_{s^p_t}) (h_d q_t + d_t) = 0 . \tag{31.2}
$$

Equation [(31.2)](#equation-pr-euler) is the first-order condition of a planner who chooses $ \{q_t\} $ to minimize

$$
\mathbb{E}_0 \sum_{t=0}^\infty \beta^t \Big\{ \tfrac{h_s}{2} q_t^2 + \tfrac{g_s}{2} (q_t - q_{t-1})^2 + (s_t + \nu_t) q_t
+ (1 + \phi_{s^p_t}) \big( \tfrac{h_d}{2} q_t^2 + d_t q_t \big) \Big\} .
$$

The planner weights the area under the inverse demand curve at date $ t $ by $ 1 + \phi_{s^p_t} $.

The weight enters only the date-$ t $ term because demand is static.

With $ g_d > 0 $ the demander’s benefit at date $ t $ depends on $ q_t, \ldots, q_{t-4} $ through the polynomial $ a(L) $ of [Dynamic Supply and Demand Curves](https://python-advanced.quantecon.org/supply_demand_var.html).

The first-order condition for $ q_t $ would then collect terms dated $ t, \ldots, t+4 $, each weighted by the regime prevailing at its own date, and no single weight would apply.

With a single regime, dividing the objective by $ 1 + \phi $ restates the result of [Policy Counterfactuals from Impulse Responses](https://python-advanced.quantecon.org/policy_counterfactuals.html): a price-stabilizing rule is equivalent to suppliers with cost parameters $ h_s/(1+\phi) $ and $ g_s/(1+\phi) $ and cost shocks $ s_t/(1+\phi) $.

The planner’s state vector is

$$
x_t = \big( q_{t-1}, \; w_{st}, \ldots, w_{s,t-3}, \; w_{dt}, \ldots, w_{d,t-3}, \; \nu_t, \ldots, \nu_{t+H-1} \big)' ,
$$

where the last block is a queue of policy shocks that agents know at date $ t $.

The control is $ u_t = q_t $.

The one-period loss is $ x_t' R x_t + u_t' Q_i u_t + 2 u_t' N_i x_t $ in regime $ i $; only $ Q_i $ and $ N_i $ depend on the regime.

The matrix $ N_i $ is the matrix $ W_i $ of [Markov Jump Linear Quadratic Dynamic Programming](https://python-advanced.quantecon.org/markov_jump_lq.html).

In [ ]:
h_s = h_d = 1.0
g_s = 10.0
β = 1 / 1.05
B_d = np.convolve(np.convolve([1, .6], [1, .4]), [1, .2])
B_s = np.convolve(np.convolve([1, -.8], [1, .4]), [1, .2])
σ_s, σ_d = np.sqrt(0.5), 2.0

H = 60                               # length of the queue of policy shocks
n = 1 + 4 + 4 + H                    # dimension of the state
i_ws, i_wd, i_ν = 1, 5, 9            # positions of w_s,t, w_d,t, and ν_t

A = np.zeros((n, n))
A[2:5, 1:4] = np.eye(3)              # shift lagged cost shocks
A[6:9, 5:8] = np.eye(3)              # shift lagged demand shocks
A[i_ν:i_ν + H - 1, i_ν + 1:] = np.eye(H - 1)   # advance the queue of policy shocks
B = np.zeros((n, 1)); B[0, 0] = 1.0  # q_t becomes next period's q_{t-1}
C = np.zeros((n, 2)); C[i_ws, 0] = σ_s; C[i_wd, 1] = σ_d

e_q = np.zeros(n); e_q[0] = 1.0
s_sel = np.zeros(n); s_sel[1:5] = B_s
d_sel = np.zeros(n); d_sel[5:9] = B_d
ν_sel = np.zeros(n); ν_sel[i_ν] = 1.0

def loss_matrices(φ, g=g_s):
    "Q_i, R, N_i for the regime with feedback coefficient φ and supplier's adjustment cost g."
    ω = 1 + φ
    Q = np.array([[(h_s + g + ω * h_d) / 2]])
    R = g / 2 * np.outer(e_q, e_q)
    N = ((-g * e_q + s_sel + ν_sel + ω * d_sel) / 2).reshape(1, n)
    return Q, R, N

### Linked Riccati equations

The planner’s losses $ x' P_i x + \rho_i $ satisfy the linked Riccati equations of [Markov Jump Linear Quadratic Dynamic Programming](https://python-advanced.quantecon.org/markov_jump_lq.html), with $ \bar P_i = \sum_j \Pi_{ij} P_j $:

$$
P_i = R + \beta A' \bar P_i A - (\beta B' \bar P_i A + N_i)' (Q_i + \beta B' \bar P_i B)^{-1} (\beta B' \bar P_i A + N_i) .
$$

The decision rule in regime $ i $ is $ q_t = -F_i x_t $ with $ F_i = (Q_i + \beta B' \bar P_i B)^{-1} (\beta B' \bar P_i A + N_i) $.

We code the iteration directly.

In [ ]:
def solve_regimes(φs, Π, gs=None, tol=1e-12, max_iter=20_000, return_P=False):
    """
    Decision rules F_i for regimes with feedback coefficients φs, adjustment costs gs,
    and transition matrix Π; with return_P, also the value matrices P_i.
    """
    m = len(φs)
    gs = [g_s] * m if gs is None else gs
    mats = [loss_matrices(φ, g) for φ, g in zip(φs, gs)]
    P = np.zeros((m, n, n))
    for _ in range(max_iter):
        P_new = np.empty_like(P)
        for i, (Q, R, N) in enumerate(mats):
            P_bar = np.tensordot(Π[i], P, axes=1)
            G = β * B.T @ P_bar @ A + N
            P_new[i] = (R + β * A.T @ P_bar @ A
                        - G.T @ np.linalg.solve(Q + β * B.T @ P_bar @ B, G))
        converged = np.max(np.abs(P_new - P)) < tol
        P = P_new
        if converged:
            break
    F = np.empty((m, 1, n))
    for i, (Q, R, N) in enumerate(mats):
        P_bar = np.tensordot(Π[i], P, axes=1)
        F[i] = np.linalg.solve(Q + β * B.T @ P_bar @ B, β * B.T @ P_bar @ A + N)
    return (F, P) if return_P else F

Quantity, price, and the tax are linear functions of $ x_t $ in each regime: $ q_t = -F_{s^p_t} x_t $, $ p_t = -d_t - h_d q_t $, and $ \tau_t = -\phi_{s^p_t} p_t + \nu_t $.

We check that the computed decision rules satisfy the supplier’s Euler equation [(31.2)](#equation-pr-euler) along a simulated path.

The conditional expectation is $ \mathbb{E}_t q_{t+1} = -\sum_k \Pi_{s^p_t k} F_k (A - B F_{s^p_t}) x_t $.

In [ ]:
def euler_residual(F, φs, Π, T_sim=300, seed=0):
    "Largest violation of the supplier's Euler equation along a simulated path."
    rng = np.random.default_rng(seed)
    m = len(φs)
    x, s, worst = np.zeros(n), 0, 0.0
    for t in range(T_sim):
        q = -(F[s] @ x).item()
        p = -d_sel @ x - h_d * q
        x_mean = (A - B @ F[s]) @ x
        Eq_next = sum(Π[s, k] * -(F[k] @ x_mean).item() for k in range(m))
        lhs = (1 + φs[s]) * p - x[i_ν]
        rhs = s_sel @ x + h_s * q + g_s * ((1 + β) * q - x[0] - β * Eq_next)
        worst = max(worst, abs(lhs - rhs))
        x = x_mean + C @ rng.standard_normal(2)
        s = rng.choice(m, p=Π[s])
    return worst

φs = np.array([0.0, 2.0])            # laissez-faire and price stabilization
Π = np.array([[0.95, 0.05],
              [0.05, 0.95]])
F = solve_regimes(φs, Π)
print(f"switching regimes: max Euler residual = {euler_residual(F, φs, Π):.1e}")

The decision rules satisfy the Euler equation to machine precision.

Expected responses to a shock condition on the regime at the date of the shock.

Let $ M_t(j) = \mathbb{E}[x_t \mathbb{1}\{s^p_t = j\} \mid x_0, s^p_0] $.

These conditional means satisfy $ M_{t+1}(k) = \sum_j \Pi_{jk} (A - B F_j) M_t(j) $, and the expected outcomes are sums over $ j $ of linear functions of $ M_t(j) $.

In [ ]:
one = np.array([[1.0]])              # transition matrix of a single regime

def expected_responses(F, φs, Π, x0, s0, T=40):
    "E[(q_t, p_t, τ_t) | x_0, s_0] for t = 0, ..., T-1."
    m = len(φs)
    M = [x0 * (j == s0) for j in range(m)]
    out = np.zeros((T, 3))
    for t in range(T):
        q_j = [-(F[j] @ M[j]).item() for j in range(m)]
        p_j = [-d_sel @ M[j] - h_d * q_j[j] for j in range(m)]
        ν_t = sum(M[j][i_ν] for j in range(m))
        out[t] = sum(q_j), sum(p_j), sum(-φs[j] * p_j[j] for j in range(m)) + ν_t
        M = [sum(Π[j, k] * (A - B @ F[j]) @ M[j] for j in range(m)) for k in range(m)]
    return out

x0_demand = C @ np.array([0.0, -1.0])     # a positive demand shock of one standard deviation
x0_supply = C @ np.array([-1.0, 0.0])     # a favorable supply shock of one standard deviation
T_plot = 20
names = ["quantity $q$", "price $p$"]

With a single regime, the planner’s solution coincides with the solution that the sequence-space method of [Policy Counterfactuals from Impulse Responses](https://python-advanced.quantecon.org/policy_counterfactuals.html) delivers for our market with static demand.

We stack the supplier’s Euler equation, the static demand curve, and the rule $ \tau_t = -\phi p_t $ for $ t = 0, \ldots, 199 $ and compare the two solutions for three rules.

In [ ]:
def sequence_matrix(φ, T_seq=200, g=g_s):
    "Supply curve, demand curve, and the rule τ + φ p = ν for t < T_seq; unknowns (q, p, τ)."
    Φ_s = np.zeros((T_seq, T_seq))
    for t in range(T_seq):
        Φ_s[t, t] = h_s + g * (1 + β)
        if t >= 1:
            Φ_s[t, t - 1] = -g
        if t + 1 < T_seq:
            Φ_s[t, t + 1] = -g * β
    I, O = np.eye(T_seq), np.zeros((T_seq, T_seq))
    return np.block([[-Φ_s, I, -I],       # supply:  p - τ - Φ_s q = s
                     [h_d * I, I, O],     # demand:  p + h_d q = -d
                     [O, φ * I, I]])      # rule:    τ + φ p = ν

def sequence_space(φ, shock, T_seq=200):
    "Paths of (q, p, τ) after a favorable supply shock (0) or a positive demand shock (1)."
    s, d = np.zeros(T_seq), np.zeros(T_seq)
    if shock == 0:
        s[:4] = -σ_s * B_s
    else:
        d[:4] = -σ_d * B_d
    x = np.linalg.solve(sequence_matrix(φ, T_seq), np.concatenate([s, -d, np.zeros(T_seq)]))
    return np.column_stack([x[:T_seq], x[T_seq:2 * T_seq], x[2 * T_seq:]])

for φ in [0.0, 0.5, 2.0]:
    F_φ = solve_regimes([φ], one)
    gap = max(np.abs(expected_responses(F_φ, [φ], one, x0, 0, 40) - sequence_space(φ, k)[:40]).max()
              for k, x0 in [(0, x0_supply), (1, x0_demand)])
    print(f"φ = {φ}: max |planner - sequence space| = {gap:.1e}")

The two methods agree to machine precision.

## Regime-dependent decision rules

Regime 0 is laissez-faire, $ \phi_0 = 0 $; regime 1 stabilizes the price, $ \phi_1 = 2 $.

Each regime persists with probability 0.95.

We compare the switching economy with two economies in which one rule holds forever.

In [ ]:
F_lf = solve_regimes([0.0], one)     # laissez-faire forever
F_st = solve_regimes([2.0], one)     # stabilization forever

print("feedback of q_t on q_(t-1)")
print(f"  switching:  laissez-faire regime {-F[0, 0, 0]:.4f},  stabilization regime {-F[1, 0, 0]:.4f}")
print(f"  permanent:  laissez-faire        {-F_lf[0, 0, 0]:.4f},  stabilization        {-F_st[0, 0, 0]:.4f}")

The decision rule in each regime differs from the rule under the corresponding permanent policy.

The feedback of $ q_t $ on $ q_{t-1} $ is 0.650 in the laissez-faire regime and 0.654 under permanent laissez-faire, and 0.548 in the stabilization regime and 0.546 under permanent stabilization.

Agents in each regime attach probability 0.05 to a switch next period, and their decisions respond.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for s0, color, label in [(0, 'C0', "laissez-faire"), (1, 'C1', "stabilization")]:
    switching = expected_responses(F, φs, Π, x0_demand, s0, T_plot)
    F_perm = F_lf if s0 == 0 else F_st
    permanent = expected_responses(F_perm, [φs[s0]], one, x0_demand, 0, T_plot)
    for i, ax in enumerate(axes):
        ax.plot(switching[:, i], color=color, lw=2,
                label=f"{label} regime, switching")
        ax.plot(permanent[:, i], color=color, lw=2, ls='--',
                label=f"{label} forever")
for i, ax in enumerate(axes):
    ax.axhline(0, color='k', lw=0.5)
    ax.set_title(names[i])
    ax.set_xlabel("periods")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

On impact, a positive demand shock of one standard deviation raises quantity by 0.57 in the stabilization regime and by 0.26 in the laissez-faire regime.

The subsidy that holds down the price induces suppliers to produce more.

Under permanent rules the impact responses are 0.573 with stabilization and 0.252 with laissez-faire.

Each regime’s impact response lies between those two numbers, close to the response under the corresponding permanent rule.

The expected path averages over future switches, and decisions at each date anticipate them.

The impact response of quantity in each regime varies with the probability of a switch.

In [ ]:
switch_probs = np.linspace(0, 0.5, 11)
impact = np.zeros((len(switch_probs), 2))
for k, pr in enumerate(switch_probs):
    Π_k = np.array([[1 - pr, pr], [pr, 1 - pr]])
    F_k = solve_regimes(φs, Π_k)
    for s0 in range(2):
        impact[k, s0] = expected_responses(F_k, φs, Π_k, x0_demand, s0, 1)[0, 0]

fig, ax = plt.subplots()
ax.plot(switch_probs, impact[:, 0], 'o-', lw=2, label="laissez-faire regime")
ax.plot(switch_probs, impact[:, 1], 's-', lw=2, label="stabilization regime")
ax.set_xlabel("probability of a switch next period")
ax.set_ylabel("impact response of $q$ to a demand shock")
ax.legend()
plt.show()

As switches become more likely, the decision rules in the two regimes move toward each other.

With switching probability 0.5, the impact responses are 0.33 and 0.52; with permanent regimes, they are 0.25 and 0.57.

The responses do not converge.

With switching probability 0.5, the two rows of $ \Pi $ are identical, so next period’s regime, and with it the value agents attach to next period’s state, does not depend on today’s regime: $ \bar P_0 = \bar P_1 $.

The decision rules $ F_0 $ and $ F_1 $ then differ only through the current regime’s $ Q_i $ and $ N_i $, so the whole gap between 0.33 and 0.52 is the effect of the rule in force today.


<a id='pr-levels'></a>

## Two levels of the Lucas critique

The switching economy distinguishes two kinds of policy change.

A switch from one regime to another of a known process leaves the decision rules $ F_0, F_1 $ unchanged.

An econometrician who conditions on the regime can estimate those rules from history and use them to predict outcomes after a switch.

That is the setting that Sims [[1982](https://python-advanced.quantecon.org/zreferences.html#id344)] regarded as typical of policy analysis.

A change in the process — the menu $ \{\phi_i\} $ or the transition matrix $ \Pi $ — changes the decision rules in every regime.

That is the setting of Lucas [[1976](https://python-advanced.quantecon.org/zreferences.html#id336)].

A reform that may arrive illustrates the second kind of change.

The economy starts in the laissez-faire regime; each period a reform to the stabilization rule occurs with probability $ r $, and the reform is permanent.

In [ ]:
print(" r      feedback on q_(t-1)   impact q   q one period later")
for r in [0.0, 0.05, 0.2]:
    Π_r = np.array([[1 - r, r], [0.0, 1.0]])
    F_r = solve_regimes(φs, Π_r)
    resp = expected_responses(F_r, φs, Π_r, x0_demand, 0, 2)
    print(f"{r:4.2f}   {-F_r[0, 0, 0]:20.4f}   {resp[0, 0]:8.4f}   {resp[1, 0]:17.4f}")

The prospect of a reform changes behavior under laissez-faire before any reform occurs.

With $ r = 0.2 $ the feedback of $ q_t $ on $ q_{t-1} $ falls from 0.654 to 0.641, and the impact response of quantity to a demand shock rises from 0.252 to 0.285.

An econometrician who estimated the laissez-faire decision rule from a period in which no reform was in prospect would mispredict behavior once a reform became possible.

A change in the process can also change whether an equilibrium is unique.

Davig and Leeper [[2007](https://python-advanced.quantecon.org/zreferences.html#id347)] show that, with Markov-switching interest-rate rules, the conditions for a unique bounded equilibrium depend on the whole switching process.

Our planning problem has a strictly convex objective, and every rule and every switching process in this lecture delivers a unique equilibrium.

[Policy Counterfactuals in a New Keynesian Economy](https://python-advanced.quantecon.org/policy_counterfactuals_nk.html) studies a rule that does not, an interest-rate peg in a New Keynesian economy, and shows that McKay and Wolf’s sufficient statistics deliver every equilibrium under it.


<a id='pr-var'></a>

## Vector autoregressions fit to regime-switching data

Because the switching economy is covariance stationary, $ \{(q_t, p_t)\} $ has an infinite-order vector autoregression with constant coefficients: the population linear least-squares projection of $ (q_t, p_t) $ on its entire past.

Even with a single rule, the autoregression has infinitely many lags, because the shocks are moving averages, and its innovations are not the structural shocks, because under either of our two rules the supply shock is not fundamental for $ (q_t, p_t) $, as Exercise 31.3 verifies.

Switching adds a third reason: the history of $ (q_t, p_t) $ also carries information about the regime.

The innovations of the autoregression are serially uncorrelated, but they are not independent of the past: they are conditionally heteroskedastic, and nonlinear functions of the past, such as the probabilities of the regimes inferred by the filter of Hamilton [[1989](https://python-advanced.quantecon.org/zreferences.html#id346)], help predict them.

We simulate long samples from the switching economy and from the two economies in which one rule holds forever.

The simulations set the policy shocks $ \nu_t $ to zero.

The econometrician fits autoregressions of orders 1, 4, and 8 to $ (q_t, p_t) $ and ignores the tax rate, which would reveal the regime.

In [ ]:
def simulate(F, φs, Π, T_sim, seed):
    "Simulate (q_t, p_t) and the regime."
    rng = np.random.default_rng(seed)
    m = len(φs)
    A_F = [A - B @ F[j] for j in range(m)]
    qp = np.zeros((T_sim, 2))
    regime = np.zeros(T_sim, dtype=int)
    x, s = np.zeros(n), 0
    for t in range(T_sim):
        q = -(F[s] @ x).item()
        qp[t] = q, -d_sel @ x - h_d * q
        regime[t] = s
        x = A_F[s] @ x + C @ rng.standard_normal(2)
        s = rng.choice(m, p=Π[s])
    return qp, regime

def var_fit(y, lags):
    "Least-squares autoregression of y_t on y_(t-1), ..., y_(t-lags): coefficients and residuals."
    Y = y[lags:]
    X = np.hstack([y[lags - k:len(y) - k] for k in range(1, lags + 1)])
    coef, *_ = np.linalg.lstsq(X, Y, rcond=None)
    return coef, Y - X @ coef

T_sim = 200_000
qp, regime = simulate(F, φs, Π, T_sim, seed=1234)
qp_lf, _ = simulate(F_lf, [0.0], one, T_sim, seed=1234)
qp_st, _ = simulate(F_st, [2.0], one, T_sim, seed=1234)

print("residual variance of q")
print("lags   switching   laissez-faire forever   stabilization forever")
for lags in [1, 4, 8]:
    v = [var_fit(y, lags)[1][:, 0].var() for y in (qp, qp_lf, qp_st)]
    print(f"{lags:4d}   {v[0]:9.4f}   {v[1]:21.4f}   {v[2]:21.4f}")

_, U = var_fit(qp, 8)
reg = regime[8:]
print(f"\nswitching economy, order 8: residual variance of q in the laissez-faire regime "
      f"{U[reg == 0, 0].var():.4f}, in the stabilization regime {U[reg == 1, 0].var():.4f}")

From order 1 to order 8, the residual variance of quantity falls by 20 percent in the switching economy, 16 percent under permanent laissez-faire, and 17 percent under permanent stabilization.

Most of the gain reflects the moving-average shocks, which are present with a single rule.

In the switching economy, the residual variance of quantity is more than four times larger in the stabilization regime than in the laissez-faire regime.

Within the laissez-faire regime, the residual variance of the pooled autoregression, 0.074, exceeds the residual variance of an autoregression fit to data from an economy that stays in laissez-faire forever, 0.065.

To separate the sources of that gap, we use the tax rate to classify the sample by regime.

We keep observations whose eight lags lie in the same regime as the current observation, and fit a separate autoregression of order 8 within each regime.

In [ ]:
lags = 8
Y = qp[lags:]
X = np.hstack([qp[lags - k:len(qp) - k] for k in range(1, lags + 1)])
same = np.array([np.all(regime[t - lags:t + 1] == regime[t]) for t in range(lags, T_sim)])
U_pooled = var_fit(qp, lags)[1]
for j, label in [(0, "laissez-faire"), (1, "stabilization")]:
    keep = same & (reg == j)
    coef_j, *_ = np.linalg.lstsq(X[keep], Y[keep], rcond=None)
    U_j = Y[keep] - X[keep] @ coef_j
    print(f"{label:14s}: residual variance of q, regime-specific {U_j[:, 0].var():.4f}, "
          f"pooled {U_pooled[keep, 0].var():.4f}")

In the laissez-faire regime, dropping observations whose lags straddle a switch lowers the pooled residual variance from 0.074 to 0.072.

Fitting separate coefficients lowers it further to 0.070.

The remaining gap to 0.065 reflects a laissez-faire decision rule that anticipates switches.

The impact response of quantity to a demand shock is 0.260 in the laissez-faire regime of the switching economy and 0.252 under permanent laissez-faire, and the squared ratio of the two, about 1.07, matches the ratio of the residual variances, 0.070 / 0.065.

In both regimes, fitting separate coefficients rather than the pooled ones, on the same observations, lowers the residual variance of quantity by less than 3 percent: from 0.0718 to 0.0699 under laissez-faire and from 0.3249 to 0.3232 under stabilization.

The autoregression’s innovation in the price, orthogonalized by ordering the price first, is mostly a demand shock.

On impact, a demand shock of one standard deviation moves the price by 1.74 in the laissez-faire regime and by 1.43 in the stabilization regime; a supply shock of one standard deviation moves it by only $ -0.03 $.

In [ ]:
for s0, label in [(0, "laissez-faire"), (1, "stabilization")]:
    p_d = expected_responses(F, φs, Π, x0_demand, s0, 1)[0, 1]
    p_s = expected_responses(F, φs, Π, x0_supply, s0, 1)[0, 1]
    print(f"{label:14s} regime: impact on p of a demand shock {p_d:.2f}, of a supply shock {p_s:.2f}")

We compare the response of quantity to that innovation with the responses of quantity to a demand shock in each regime, each scaled to raise the price by one unit on impact.

In [ ]:
def var_price_response(y, lags=8):
    "Response of (q, p) to the price innovation of an autoregression, price ordered first, unit impact on p."
    coef, U = var_fit(y, lags)
    B_var = [coef[2 * k:2 * k + 2].T for k in range(lags)]      # y_t = Σ_k B_k y_(t-k) + u_t
    Ψ = [np.eye(2)]
    for h in range(1, T_plot):
        Ψ.append(sum(B_var[k] @ Ψ[h - 1 - k] for k in range(min(lags, h))))
    swap = np.array([[0, 1], [1, 0]])
    L_p = swap @ np.linalg.cholesky(swap @ np.cov(U.T) @ swap) @ swap
    resp = np.array([Ψ_h @ L_p[:, 1] for Ψ_h in Ψ])
    return resp / resp[0, 1]

var_resp = var_price_response(qp)

fig, ax = plt.subplots()
for s0, color, label in [(0, 'C0', "laissez-faire regime"), (1, 'C1', "stabilization regime")]:
    resp = expected_responses(F, φs, Π, x0_demand, s0, T_plot)
    ax.plot(resp[:, 0] / resp[0, 1], color=color, lw=2, label=label)
ax.plot(var_resp[:, 0], 'k--', lw=2, label="autoregression, switching data")
for y_perm, color, label in [(qp_lf, 'C0', "laissez-faire forever"), (qp_st, 'C1', "stabilization forever")]:
    ax.plot(var_price_response(y_perm)[:, 0], ':', color=color, lw=2,
            label=f"autoregression, data from {label}")
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel("periods")
ax.set_ylabel("response of $q$ per unit impact on $p$")
ax.legend()
plt.show()

On impact, the autoregression’s response is 0.25 per unit of price, against 0.15 in the laissez-faire regime and 0.40 in the stabilization regime.

A constant-coefficient autoregression averages over regimes.

The same autoregression fit to data from an economy that stays in one regime forever recovers the response to a demand shock in that economy almost exactly: 0.144 per unit of price against a true 0.144 under laissez-faire, and 0.400 against 0.401 under stabilization.

Within one regime, the price innovation is mostly a demand shock, and the demand shock is fundamental, so ordering the price first approximately identifies it.

The error of the pooled autoregression therefore comes from averaging over regimes, not from the identification scheme.

In the passage quoted in the overview, Sims and Zha [[2006](https://python-advanced.quantecon.org/zreferences.html#id345)] conjecture that the public’s behavior can be well approximated as linear and time invariant even when policy switches.

As a statement about fit, our results bear the conjecture out: the constant-coefficient autoregression fits almost as well as autoregressions that condition on the regime.

As a statement about the private sector’s equations, they bear it out only when regimes are drawn afresh each period, or nearly so.

The supply and demand curves are linear and time invariant by construction, but the supply curve contains the expectation $ \mathbb{E}_t q_{t+1} $.

A model like Sims and Zha’s holds fixed the private sector’s equations written in observable variables, after expectations are replaced by forecasts from current and past variables.

In regime $ i $, the forecast of $ q_{t+1} $ puts weight $ c_i $ on $ q_t $ and weight $ a_i $ on the current demand innovation $ w_{dt} $; both depend on the regime only through suppliers’ forecast of next period’s decision rule, $ \bar F_i = \sum_k \Pi_{ik} F_k $.

Because demand is static, $ d_t = -(p_t + h_d q_t) $, and $ d_t = w_{dt} $ plus lagged innovations.

Substituting the forecast into the supply curve and replacing the demand shock with prices and quantities gives

$$
a_{p,i} \, p_t - \tau_t = a_{q,i} \, q_t + (\text{lagged variables and current and lagged cost innovations}),
$$

with

$$
a_{q,i} = h_s + g_s (1 + \beta) - \beta g_s (c_i - h_d a_i), \qquad a_{p,i} = 1 - \beta g_s a_i .
$$

The weight $ a_i $ is negative, because a positive demand shock is a negative $ w_{dt} $, so $ a_{p,i} > 1 $.

We compute these coefficients for the switching chain of this lecture, for the two permanent rules, and for a chain whose regime is drawn afresh each period, with switching probability 0.5.

In [ ]:
def supply_coefficients(F_, Π_):
    "Coefficients (a_q, a_p) of the supply curve a_p p_t - τ_t = a_q q_t + ... in each regime."
    out = []
    for i in range(2):
        F_bar = Π_[i, 0] * F_[0] + Π_[i, 1] * F_[1]     # E_t q_(t+1) = -F_bar (A x_t + B q_t)
        c_i = -(F_bar @ B)[0, 0]                         # weight on q_t
        a_i = -(F_bar @ A)[0, i_wd]                      # weight on the demand innovation w_dt
        out.append((h_s + g_s * (1 + β) - β * g_s * (c_i - h_d * a_i), 1 - β * g_s * a_i))
    return out

Π_iid = np.full((2, 2), 0.5)
cases = [("switching chain of this lecture", F, Π),
         ("permanent rules", np.stack([F_lf[0], F_st[0]]), np.eye(2)),
         ("regimes drawn afresh each period", solve_regimes(φs, Π_iid), Π_iid)]
print(" " * 34 + "laissez-faire       stabilization")
print(" " * 34 + " a_q     a_p         a_q     a_p")
for label, F_, Π_ in cases:
    (q_0, p_0), (q_1, p_1) = supply_coefficients(F_, Π_)
    print(f"{label:33s} {q_0:6.2f}  {p_0:5.2f}       {q_1:6.2f}  {p_1:5.2f}")

With regimes as persistent as ours, the coefficient on quantity is nearly the same in the two regimes, but the coefficient on the price, relative to the tax, is about 60 percent larger under stabilization, close to its values under the permanent rules.

Under stabilization, the tax falls when a demand shock raises the price, so quantity responds more strongly to the shock, next period as well as on impact; suppliers, who forecast that response and face adjustment costs, respond more strongly to today’s price, which together with quantity reveals the demand shock.

The supply curve written in observables thus switches with the policy regime, as the Lucas critique says it should.

With regimes drawn afresh each period, suppliers’ forecast of next period’s rule does not depend on today’s regime, and the supply curve is the same in both regimes, exactly as Sims and Zha assume, although the decision rules still differ (Fig. 31.2).

So, as a statement about the private sector’s equations, the conjecture holds exactly only when regimes are drawn afresh each period, and it fails in our market, whose regimes last 20 periods on average.

One caveat concerns normalization.

Without shocks to the tax rule, $ \tau_t = 0 $ under laissez-faire, so the coefficient on $ \tau_t $ is not identified there, and adding a multiple of the stabilization rule $ \tau_t + 2 p_t = 0 $ to the stabilization regime’s supply curve and rescaling could make its contemporaneous coefficients match those under laissez-faire.

With an i.i.d. shock in the tax rule, as in Sims and Zha’s policy equation, the form above is the only combination of the supply curve and the rule whose disturbance is uncorrelated with that shock, because the forecast of $ q_{t+1} $ puts no weight on $ \nu_t $; and the forecast’s weights on lagged demand innovations, which adding the tax rule cannot change because the rule has no lags, also differ across regimes, by about the same factor as $ a_i $.

The constant-coefficient autoregression for $ (q_t, p_t) $ has a further problem: it omits the tax and pools two policy rules, so that its impulse response describes behavior in neither regime.

Sims and Zha [[2006](https://python-advanced.quantecon.org/zreferences.html#id345)] let the variances of the disturbances switch in every equation and, in some specifications, let the coefficients of the policy equation switch while those of the other equations stay fixed; their best-fitting model lets only the variances switch, and versions in which other coefficients also switch fit worse.

In our market, a model that held the private sector’s equations fixed while only the policy equation switched would be misspecified, because the supply equation’s coefficients on the price, relative to the tax, and on lagged variables switch too; a Markov-switching vector autoregression in the spirit of Hamilton [[1989](https://python-advanced.quantecon.org/zreferences.html#id346)], with all coefficients switching, would capture the regime-dependent responses that the constant-coefficient autoregression averages away.

That the constant-coefficient autoregression fits almost as well as autoregressions that condition on the regime suggests that, in samples of realistic length, comparisons of fit have little power to detect regime-dependent coefficients in the private sector’s equations.


<a id='pr-scenarios'></a>

## Scenarios and regime changes

McKay and Wolf’s counterfactual is the stabilization rule announced at date 0 and kept forever.

In a regime-switching economy, a reform to the stabilization rule may be reversed.

We let the stabilization regime end with probability $ e $ each period and compute the expected response to a demand shock that occurs in the period of the reform.

A reversal is permanent: once laissez-faire returns, it prevails forever.

(Making the reversal permanent keeps small the tree of regime histories that we use [below](#pr-tree).)

In [ ]:
fig, ax = plt.subplots()
for e, ls in [(0.0, '-'), (0.05, '--'), (0.2, '-.'), (0.5, ':')]:
    Π_e = np.array([[1.0, 0.0], [e, 1 - e]])     # laissez-faire is absorbing
    F_e = solve_regimes(φs, Π_e)
    resp = expected_responses(F_e, φs, Π_e, x0_demand, 1, T_plot)
    label = "permanent reform" if e == 0 else f"exit probability {e}"
    ax.plot(resp[:, 0], lw=2, ls=ls, color='C1', label=label)
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel("periods")
ax.set_ylabel("expected response of $q$")
ax.legend()
plt.show()

On impact and in the two periods that follow, the more likely a reversal, the smaller the response of quantity.

With exit probability 0.2, the response of quantity one period after the shock is about 7 percent smaller than under a permanent reform; with exit probability 0.5, it is about 18 percent smaller.

Two forces produce the gap.

Agents who expect a reversal adjust less to the reformed rule; the expected path also averages over reversals that occur.

From period 4 on (the impact period is period 0), the response under every reversible reform exceeds the response under the permanent reform, because a reversal returns the economy to laissez-faire, under which quantity adjusts more slowly.

McKay and Wolf’s scenario corresponds to $ e = 0 $.

[A counterfactual rule that may be reversed](#pr-tree) shows that their sufficient statistics also deliver the reversible reforms with $ e > 0 $.

The dynamic programming squared lectures, such as [Sustainable Plans for a Calvo Model](https://python-advanced.quantecon.org/calvo_abreu.html) and [Accounting for Monetary and Fiscal Policy](https://python-advanced.quantecon.org/dovis_accounting_mf.html), study what determines the credibility that this lecture summarizes by $ e $.

## McKay and Wolf in a regime-switching economy

Suppose that the historical data come from the switching economy.

An econometrician estimates expected responses to the demand shock and to contemporaneous and news shocks to the tax rule, conditional on the regime at the date of the shock.

McKay and Wolf’s construction from [Policy Counterfactuals from Impulse Responses](https://python-advanced.quantecon.org/policy_counterfactuals.html) uses these expected responses.

In this section and the next, $ x $ denotes paths of the outcomes $ (q, p, \tau) $, as in [Policy Counterfactuals from Impulse Responses](https://python-advanced.quantecon.org/policy_counterfactuals.html), and $ x_A(\varepsilon) $ the expected response of those outcomes to a non-policy shock $ \varepsilon $, here the demand shock; the code still computes these paths from the planner’s state, starting from vectors such as `x0_demand`.

The supplier’s Euler equation and the demand curve are linear with coefficients that do not depend on the regime.

By the law of iterated expectations, expected paths of $ (q, p, \tau) $ satisfy those equations.

A counterfactual rule $ \tau_t = -\tilde \phi p_t $ that holds forever is a linear restriction on expected paths.

McKay and Wolf’s construction recovers the permanent counterfactual exactly.

In [ ]:
T_mw = 40

def stacked(resp):
    return np.concatenate([resp[:, 0], resp[:, 1], resp[:, 2]])

def mw_from_switching(F, φs, Π, s0, φ_new):
    "McKay–Wolf counterfactual from expected responses in the switching economy."
    x_A = stacked(expected_responses(F, φs, Π, x0_demand, s0, T_mw))
    Θ = np.column_stack([stacked(expected_responses(F, φs, Π, np.eye(n)[i_ν + ℓ], s0, T_mw))
                         for ℓ in range(T_mw)])
    R_new = np.hstack([np.zeros((T_mw, T_mw)), φ_new * np.eye(T_mw), np.eye(T_mw)])
    ν = np.linalg.solve(R_new @ Θ, -R_new @ x_A)
    return (x_A + Θ @ ν).reshape(3, T_mw).T, x_A.reshape(3, T_mw).T

truth = expected_responses(F_st, [2.0], one, x0_demand, 0, T_mw)
for s0, label in [(0, "laissez-faire"), (1, "stabilization")]:
    cf, x_A = mw_from_switching(F, φs, Π, s0, 2.0)
    print(f"data from the {label} regime: baseline impact on q {x_A[0, 0]:.4f}, "
          f"max |McKay–Wolf - permanent stabilization| = {np.abs(cf[:30] - truth[:30]).max():.1e}")

The baseline responses depend on the regime at the date of the shock.

From either regime, the construction recovers the response under a permanent stabilization rule to machine precision.

A switching historical rule does not impair McKay and Wolf’s construction.


<a id='pr-tree'></a>

## A counterfactual rule that may be reversed

Under the reversible reform of [Scenarios and regime changes](#pr-scenarios), the expected tax is $ \mathbb{E}_0 \tau_t = -\sum_j \phi_j \mathbb{E}_0 [p_t \mathbb{1}\{s^p_t = j\}] + \mathbb{E}_0 \nu_t $.

That restriction involves the joint distribution of prices and regimes, so it is not a linear restriction on date-0 expected paths.

Caravello *et al.* [[2026](https://python-advanced.quantecon.org/zreferences.html#id351)] show that it is a linear restriction on paths that are contingent on the *history* of regimes, and that McKay and Wolf’s sufficient statistics deliver those paths.

Their construction works on the tree of regime histories.

At each node of the tree, the supplier’s Euler equation and the demand curve hold, with expectations conditioned on the node and computed with the transition probabilities.

Because these equations are linear, the outcome at each node equals the baseline response to the shock plus the responses to all the news about policy that has arrived along the history leading to the node.

Write $ \eta_t $ for a history of regimes through date $ t $ and $ \eta_u(\eta_t) $ for its first $ u + 1 $ entries, and let the policy shocks $ \nu $ depend on the history.

The outcome at $ \eta_t $ is


<a id='equation-pr-tree'></a>
$$
x_t(\eta_t) = x_{A,t}(\varepsilon) + \sum_{u=0}^t \sum_{\ell \ge u} \Theta_\nu(t-u, \ell-u) \Big( \mathbb{E}[\nu_\ell \mid \eta_u(\eta_t)] - \mathbb{E}[\nu_\ell \mid \eta_{u-1}(\eta_t)] \Big) , \tag{31.3}
$$

where $ \Theta_\nu(t, \ell) $ is the response at date $ t $ to news, received at date 0, of a unit shock to the tax rule at date $ \ell $, and the expectation conditioned on $ \eta_{-1} $ is zero.

The formula uses the fact that the private sector’s equations do not change over time: news that arrives at date $ u $ has the effects of news that arrives at date 0, shifted $ u $ periods.

The econometrician chooses the history-contingent shocks so that the counterfactual rule holds at every node.

She needs the objects she used before, $ x_A(\varepsilon) $ and $ \Theta_\nu $, measured in the economy that generated the data, together with the transition probabilities that define the counterfactual.

She does not need a model of the private sector.

### A reversible reform

For our reform, the tree is small.

Let $ \mathcal N_t $ be the node at which the reform is still in place at date $ t $, and $ \mathcal R_{k,t} $ the node at which it was reversed at date $ k $, where $ 1 \le k \le t $.

Node $ \mathcal N_t $ has probability $ (1-e)^t $, and the reversal occurs at date $ k $ with probability $ (1-e)^{k-1} e $.

After a reversal no uncertainty remains, so news arrives only at the nodes $ \mathcal N_u $ and at the nodes $ \mathcal R_{u,u} $ at which a reversal occurs.

Let $ \xi_u $ be the news about $ (\nu_u, \nu_{u+1}, \ldots) $ that arrives at $ \mathcal N_u $ and, for $ u \ge 1 $, let $ \chi_u $ be the news that arrives at $ \mathcal R_{u,u} $.

The date-0 news $ \xi_0 $ is the initial expected path of the policy shocks.

For $ u \ge 1 $, news cannot be predictable, so $ (1-e)\, \xi_u + e\, \chi_u = 0 $: agents revise their forecasts of future policy shocks in one direction when the reform survives and in the other when it is reversed.

The unknowns are the news vectors $ \xi_0, \xi_1, \ldots $, and the equations are the rule $ \tau_t = -2 p_t $ at every node $ \mathcal N_t $ and the rule $ \tau_t = 0 $ at every node $ \mathcal R_{k,t} $.

Caravello *et al.* [[2026](https://python-advanced.quantecon.org/zreferences.html#id351)] take $ x_A(\varepsilon) $ and $ \Theta_\nu $ from an economy with a fixed baseline rule.

Expected responses in the switching economy serve as well: they are deterministic sequences that satisfy the private sector’s equations, which is all the construction uses.

As in the previous section, we take them from the switching economy, conditional on the laissez-faire regime at the date of the shock.

In [ ]:
T_tree = 60

def news_responses(F, φs, Π, s0, T):
    "Stacked expected responses of (q, p, τ) to date-0 news of a unit tax-rule shock at each date ℓ < T."
    return np.column_stack([stacked(expected_responses(F, φs, Π, np.eye(n)[i_ν + ℓ], s0, T))
                            for ℓ in range(T)])

def mw_tree(x_A, Θ, φ_start, φ_after, e, T=T_tree):
    """
    McKay–Wolf counterfactual on the tree of regime histories.

    The rule τ = -φ_start p holds at date 0; at each date k ≥ 1, if it has not
    already happened, a permanent switch to τ = -φ_after p occurs with probability e.
    x_A (length 3T) and Θ (3T x T) stack the paths of (q, p, τ).

    Returns the expected paths, the paths at the nodes N_t (no switch yet), and the
    paths after a switch at each date k = 1, ..., T-1.
    """
    # the unknowns ξ_u(ℓ), ℓ = u, ..., T-1, occupy positions start[u], ..., start[u + 1] - 1
    start = np.concatenate([[0], np.cumsum(T - np.arange(T))])
    def rule_rows(φ):
        "Responses of τ_t + φ p_t at date t to date-0 news about ν_ℓ."
        return Θ[2 * T:] + φ * Θ[T:2 * T]
    def add_news(row, Θφ, t, u, weight):
        "Add the effect on date t of the news ξ_u, scaled by weight."
        row[start[u]:start[u + 1]] += weight * Θφ[t - u, :T - u]
    Θ_start, Θ_after = rule_rows(φ_start), rule_rows(φ_after)
    rows, rhs = [], []
    for t in range(T):                         # the rule at N_t
        row = np.zeros(start[-1])
        for u in range(t + 1):
            add_news(row, Θ_start, t, u, 1.0)
        rows.append(row); rhs.append(-(x_A[2 * T + t] + φ_start * x_A[T + t]))
    for k in range(1, T):                      # the rule at R_{k,t}
        for t in range(k, T):
            row = np.zeros(start[-1])
            for u in range(k):
                add_news(row, Θ_after, t, u, 1.0)
            add_news(row, Θ_after, t, k, -(1 - e) / e)     # χ_k = -(1 - e) / e ξ_k
            rows.append(row); rhs.append(-(x_A[2 * T + t] + φ_after * x_A[T + t]))
    sol = np.linalg.solve(np.array(rows), np.array(rhs))

    def effect(u, weight):
        "Paths of (q, p, τ) that the news ξ_u, scaled by weight, produces."
        news = np.zeros(T)
        news[u:] = sol[start[u]:start[u + 1]]
        out = np.zeros((T, 3))
        for b in range(3):
            out[u:, b] = weight * (Θ[b * T:(b + 1) * T - u, :T - u] @ news[u:])
        return out
    base = x_A.reshape(3, T).T
    effects = np.cumsum([effect(u, 1.0) for u in range(T)], axis=0)
    path_N = base + effects[-1]
    paths_R = np.array([base + effects[k - 1] + effect(k, -(1 - e) / e) for k in range(1, T)])
    expected = np.zeros((T, 3))
    for t in range(T):
        expected[t] = (1 - e)**t * path_N[t]
        for k in range(1, t + 1):
            expected[t] += (1 - e)**(k - 1) * e * paths_R[k - 1, t]
    return expected, path_N, paths_R

x_A_tree = stacked(expected_responses(F, φs, Π, x0_demand, 0, T_tree))
Θ_tree = news_responses(F, φs, Π, 0, T_tree)

for e in [0.05, 0.2, 0.5]:
    Π_e = np.array([[1.0, 0.0], [e, 1 - e]])
    F_e = solve_regimes(φs, Π_e)
    truth_e = expected_responses(F_e, φs, Π_e, x0_demand, 1, 30)
    tree_e, _, _ = mw_tree(x_A_tree, Θ_tree, 2.0, 0.0, e)
    print(f"exit probability {e}: max |tree - Markov jump LQ| over 30 periods = "
          f"{np.abs(tree_e[:30] - truth_e).max():.1e}")

From data on the switching economy, the tree construction reproduces the reversible reform to machine precision.

### Three checks

The comparison so far concerns expected paths, but the construction delivers more: a path for every history.

Christian Wolf and Alisdair McKay kindly shared a notebook that implements this construction for our market, and we borrow three of its checks.

The first compares every branch of the tree with the path that the Markov jump decision rules produce along that realized history of regimes.

Along a realized history, agents still use the decision rules $ F_j $ computed from the transition matrix, because they do not know the future of the regime.

In [ ]:
def realized_path(F, φs, regimes, x0):
    "Outcomes (q, p, τ) along a given sequence of regimes, under decision rules F that anticipate switches."
    x = x0.copy()
    out = np.zeros((len(regimes), 3))
    for t, j in enumerate(regimes):
        q = -(F[j] @ x).item()
        p = -d_sel @ x - h_d * q
        out[t] = q, p, -φs[j] * p + x[i_ν]
        x = (A - B @ F[j]) @ x
    return out

e, T_c = 0.2, 30
Π_e = np.array([[1.0, 0.0], [e, 1 - e]])
F_e = solve_regimes(φs, Π_e)
_, path_N, paths_R = mw_tree(x_A_tree, Θ_tree, 2.0, 0.0, e)
gap = np.abs(path_N[:T_c] - realized_path(F_e, φs, [1] * T_c, x0_demand)).max()
for k in range(1, T_c):
    regimes = [1] * k + [0] * (T_c - k)           # reversal at date k
    gap = max(gap, np.abs(paths_R[k - 1, k:T_c]
                          - realized_path(F_e, φs, regimes, x0_demand)[k:]).max())
print(f"max |tree - Markov jump LQ| over every branch and {T_c} periods = {gap:.1e}")

Every branch agrees.

The second check makes no use of the Markov jump solution.

At each node, it evaluates the supplier’s Euler equation and the demand curve, with the expectation of next period’s quantity computed from the two successors of the node and their probabilities.

In [ ]:
def node_residuals(path_N, paths_R, e, T_c):
    "Largest residual of the supply and demand curves at the nodes of the tree before date T_c."
    d = np.zeros(T_c + 1)
    d[:4] = -σ_d * B_d                               # the positive demand shock
    def q_at(k, t):
        "Quantity at date t on the branch with a reversal at date k (k = 0: no reversal)."
        return path_N[t, 0] if k == 0 or t < k else paths_R[k - 1, t, 0]
    worst = 0.0
    for t in range(T_c):
        for k in range(t + 1):
            q, p, τ = path_N[t] if k == 0 else paths_R[k - 1, t]
            q_lag = 0.0 if t == 0 else q_at(k, t - 1)
            if k == 0:                               # the reform survives or is reversed tomorrow
                Eq_next = (1 - e) * q_at(0, t + 1) + e * q_at(t + 1, t + 1)
            else:                                    # no uncertainty after a reversal
                Eq_next = q_at(k, t + 1)
            supply = (p - τ) - h_s * q - g_s * ((q - q_lag) - β * (Eq_next - q))
            demand = p + d[t] + h_d * q
            worst = max(worst, abs(supply), abs(demand))
    return worst

print(f"largest residual of the private sector's equations at {T_c * (T_c + 1) // 2} nodes = "
      f"{node_residuals(path_N, paths_R, e, T_c):.1e}")

The paths on the tree form an equilibrium of the economy with the reversible reform without any reference to the planning problem that produced the Markov jump solution.

The third check shows that the answer does not depend on the rule that generated the data.

We repeat the construction with $ x_A(\varepsilon) $ and $ \Theta_\nu $ computed in sequence space under permanent rules $ \phi = 0.5 $ and $ \phi = 2 $, and, under $ \phi = 0.5 $, with a tree that runs for 80 periods instead of 60.

In [ ]:
def sequence_space_objects(φ, T=T_tree, T_seq=200):
    "x_A and Θ_ν, stacked (q, p, τ) for dates t < T, under a permanent rule φ."
    rhs = np.zeros((3 * T_seq, T + 1))
    rhs[T_seq:T_seq + 4, 0] = σ_d * B_d           # a positive demand shock: -d = σ_d B_d
    rhs[2 * T_seq + np.arange(T), 1 + np.arange(T)] = 1.0
    sol = np.linalg.solve(sequence_matrix(φ, T_seq), rhs)
    rows = np.concatenate([b * T_seq + np.arange(T) for b in range(3)])
    return sol[rows, 0], sol[rows, 1:]

truth_e = expected_responses(F_e, φs, Π_e, x0_demand, 1, T_c)
for label, T_b, (x_A_b, Θ_b) in [
        ("switching economy, laissez-faire regime", T_tree, (x_A_tree, Θ_tree)),
        ("permanent rule φ = 0.5", T_tree, sequence_space_objects(0.5)),
        ("permanent rule φ = 2", T_tree, sequence_space_objects(2.0)),
        ("permanent rule φ = 0.5, 80 periods", 80, sequence_space_objects(0.5, T=80))]:
    tree_b, _, _ = mw_tree(x_A_b, Θ_b, 2.0, 0.0, e, T=T_b)
    print(f"data from {label:40s}: max |tree - Markov jump LQ| = "
          f"{np.abs(tree_b[:T_c] - truth_e).max():.1e}")

Whichever rule generated the data, the construction delivers the same counterfactual.

The inputs differ across the four cases: $ \Theta_\nu $ under $ \phi = 0.5 $ is not $ \Theta_\nu $ under $ \phi = 2 $.

What they share are the effects of tax paths on quantity and price, the matrix $ \Theta_{(q,p),\tau} = \Theta_{(q,p),\nu} \Theta_{\tau,\nu}^{-1} $, which the private sector’s equations alone determine.

[The baseline rule does not matter](https://python-advanced.quantecon.org/policy_counterfactuals.html#pc-portable) in [Policy Counterfactuals from Impulse Responses](https://python-advanced.quantecon.org/policy_counterfactuals.html) defines that matrix and calls this property portability, following Caravello *et al.* [[2026](https://python-advanced.quantecon.org/zreferences.html#id351)].

### Paths along the branches

The next figure shows, for exit probability 0.2, the expected path together with the paths conditional on the date of reversal.

In [ ]:
e = 0.2
tree_e, path_N, paths_R = mw_tree(x_A_tree, Θ_tree, 2.0, 0.0, e)
Π_e = np.array([[1.0, 0.0], [e, 1 - e]])
truth_e = expected_responses(solve_regimes(φs, Π_e), φs, Π_e, x0_demand, 1, T_plot)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, (ax, title) in enumerate(zip(axes, names + ["tax $\\tau$"])):
    ax.plot(path_N[:T_plot, i], 'k:', lw=1.5, label="no reversal")
    for k in [1, 2, 4, 8]:
        branch = np.concatenate([path_N[:k, i], paths_R[k - 1, k:T_plot, i]])
        ax.plot(branch, lw=1, alpha=0.7, label=f"reversal at date {k}")
    ax.plot(truth_e[:, i], 'C1', lw=3, alpha=0.5, label="expected, Markov jump LQ")
    ax.plot(tree_e[:T_plot, i], 'ko', ms=3, label="expected, tree")
    ax.axhline(0, color='k', lw=0.5)
    ax.set_title(title)
    ax.set_xlabel("periods")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

After a reversal, quantity adjusts more slowly, as under laissez-faire, and the price responds more.

The tax panel shows what the counterfactual rule does along each branch: while the reform lasts, a subsidy while the price is high and a small tax once it falls below normal, and no tax at all once the reform is reversed.

The expected path averages these branches with the probabilities of reversal at each date.

The tree has one branch for each date of reversal because a reversal is permanent.

With a recurrent chain, such as one under which laissez-faire can again give way to stabilization, the number of histories doubles each period, and the construction is correspondingly more costly; Exercise 31.9 truncates such a tree and shows how quickly the truncated counterfactual converges.

[Policy Counterfactuals in a New Keynesian Economy](https://python-advanced.quantecon.org/policy_counterfactuals_nk.html) applies the same construction to the example of Caravello *et al.* [[2026](https://python-advanced.quantecon.org/zreferences.html#id351)], forward guidance that ends at a random date.

The extension leaves one feature of McKay and Wolf’s experiment intact.

At date 0, agents learn of a switching process that they had not anticipated, and they believe it.

In the language of [Two levels of the Lucas critique](#pr-levels), the construction handles switches within a known process together with one unanticipated change of process at date 0.

It is Lucas’s experiment applied to a process rather than to a rule, and the objection of Sargent and Wallace [[1976](https://python-advanced.quantecon.org/zreferences.html#id343)] to unanticipated changes in rules applies to that change of process.


<a id='pr-ilp'></a>

## Learning the effects of policy from regime switches

So far, McKay and Wolf’s econometrician has learned the effects of policy from responses to policy shocks, the contemporaneous and news shocks $ \nu $ to the tax rule.

Manuel and Wolf [[2026](https://python-advanced.quantecon.org/zreferences.html#id352)] show that switches in the policy rule can take the place of policy shocks.

Suppose that there are no shocks to the tax rule, $ \nu_t = 0 $, and that the econometrician observes the demand shock, say through a measured shifter of demand, and the regime, which the tax rate reveals.

She runs the *interacted local projections*


<a id='equation-pr-ilp'></a>
$$
y_{t+h} = a_h + b_h \tilde w_{dt} + c_h \zeta_t + \delta_h \, \tilde w_{dt} \zeta_t + v_{t+h}, \qquad h = 0, 1, \ldots , \tag{31.4}
$$

for $ y = q, p, \tau $, where $ \tilde w_{dt} = -w_{dt} / \sigma_d $ is a positive demand shock scaled to have unit variance, $ \zeta_t = 1 $ in the stabilization regime, and $ \zeta_t = 0 $ under laissez-faire.

Because the shock is independent of the regime, $ \delta_h $ is the difference between the expected responses to the demand shock in the two regimes, the gap between the two solid lines in Fig. 31.1.

McKay and Wolf’s Proposition 1, run in reverse, explains why that gap measures the effects of policy.

The same shock propagates differently in the two regimes only because the tax responds differently, and the private sector’s equations are the same in both regimes.

So $ \delta_\tau = (\delta_{\tau,0}, \delta_{\tau,1}, \ldots) $ is a *treatment*, a path of the tax, and $ \delta_q $ and $ \delta_p $ are its effects:


<a id='equation-pr-ilp-effects'></a>
$$
\begin{pmatrix} \delta_q \\ \delta_p \end{pmatrix} = \Theta_{(q,p),\tau} \, \delta_\tau , \tag{31.5}
$$

where $ \Theta_{(q,p),\tau} $ maps a path of the tax, known at date 0, into paths of quantity and price.

The tax paths in $ \delta_\tau $ are not known at date 0, because the regime may switch later; but, as in the construction of the permanent counterfactual above, the law of iterated expectations implies that *expected* paths of $ (q, p, \tau) $ satisfy the private sector’s equations, so $ \Theta_{(q,p),\tau} $ maps the difference in expected tax paths into the differences in expected quantity and price.

The private sector’s equations alone determine $ \Theta_{(q,p),\tau} $; it does not depend on the policy rule.

Manuel and Wolf [[2026](https://python-advanced.quantecon.org/zreferences.html#id352)] state conditions under which [(31.5)](#equation-pr-ilp-effects) holds in a large class of linear economies with stochastically and even endogenously switching policy.

The “shock” must not itself be shaped by the regime, and the regime must affect how the shock propagates only through policy.

In our market, a Markov chain independent of everything else selects the regime, and the demand shock is a primitive shock, so both conditions hold.

We simulate the switching economy with $ \nu_t = 0 $ and run the projections [(31.4)](#equation-pr-ilp).

In [ ]:
def simulate_with_shocks(F, φs, Π, T_sim, seed):
    "Simulate (q_t, p_t, τ_t), the regime, and the standardized shocks (w_s, w_d)."
    rng = np.random.default_rng(seed)
    m = len(φs)
    A_F = [A - B @ F[j] for j in range(m)]
    y = np.zeros((T_sim, 3))
    regime = np.zeros(T_sim, dtype=int)
    w = rng.standard_normal((T_sim, 2))
    x, s = C @ w[0], 0
    for t in range(T_sim):
        q = -(F[s] @ x).item()
        p = -d_sel @ x - h_d * q
        y[t] = q, p, -φs[s] * p
        regime[t] = s
        if t + 1 < T_sim:
            x = A_F[s] @ x + C @ w[t + 1]
            s = rng.choice(m, p=Π[s])
    return y, regime, w

def interacted_lp(y, shock, ζ, n_h):
    "Coefficients δ_h on shock × ζ in the projections of y_(t+h), h = 0, ..., n_h - 1."
    n_obs = len(shock)
    δ = np.zeros((n_h, y.shape[1]))
    for h in range(n_h):
        X = np.column_stack([np.ones(n_obs - h), shock[:n_obs - h], ζ[:n_obs - h],
                             shock[:n_obs - h] * ζ[:n_obs - h]])
        coef, *_ = np.linalg.lstsq(X, y[h:], rcond=None)
        δ[h] = coef[3]
    return δ

y_sim, regime_sim, w_sim = simulate_with_shocks(F, φs, Π, T_sim, seed=4321)
ζ = (regime_sim == 1).astype(float)
n_h = 12
δ_hat = interacted_lp(y_sim, -w_sim[:, 1], ζ, n_h)    # a positive demand shock is -w_d

δ_pop = {}
for name, x0 in [("demand", x0_demand), ("supply", x0_supply)]:
    δ_pop[name] = (expected_responses(F, φs, Π, x0, 1, 80)
                   - expected_responses(F, φs, Π, x0, 0, 80))

print("h    δ_τ estimate  population    δ_q estimate  population")
for h in range(6):
    print(f"{h}   {δ_hat[h, 2]:12.3f} {δ_pop['demand'][h, 2]:11.3f}   "
          f"{δ_hat[h, 0]:12.3f} {δ_pop['demand'][h, 0]:11.3f}")

With 200,000 observations, the estimates are close to their population values.

Samples of that length do not exist.

We repeat the projections on 400 samples of 300 periods each, in which the regime switches about 15 times.

In [ ]:
n_rep, T_short = 400, 300
δ_short = np.zeros((n_rep, n_h, 3))
for r in range(n_rep):
    y_r, regime_r, w_r = simulate_with_shocks(F, φs, Π, T_short, seed=10_000 + r)
    δ_short[r] = interacted_lp(y_r, -w_r[:, 1], (regime_r == 1).astype(float), n_h)
lo, med, hi = np.percentile(δ_short, [5, 50, 95], axis=0)

print("h     δ_τ:  5%  median    95%  population      δ_q:  5%  median    95%  population")
for h in range(4):
    print(f"{h}   {lo[h, 2]:10.2f} {med[h, 2]:7.2f} {hi[h, 2]:6.2f} {δ_pop['demand'][h, 2]:11.2f}   "
          f"{lo[h, 0]:10.2f} {med[h, 0]:7.2f} {hi[h, 0]:6.2f} {δ_pop['demand'][h, 0]:11.2f}")

In samples of realistic length, the projections estimate the treatment at dates 0 and 1 fairly precisely: the 90 percent band for $ \delta_{\tau,0} $ runs from $ -3.36 $ to $ -2.32 $, around a population value of $ -2.87 $.

Its effects are much less precise: the band for $ \delta_{q,0} $ runs from 0.13 to 0.48, around a population value of 0.31, and by date 3 the band for $ \delta_q $ includes zero.

Regime switches carry identifying information, but the 15 or so switches in a sample of 300 periods identify the effects of policy only roughly.

We now check [(31.5)](#equation-pr-ilp-effects) by feeding the population treatment $ \delta_\tau $ into the private sector’s equations, without any reference to a policy rule.

In [ ]:
def tax_effects(T_seq=200, g=g_s):
    "Θ_(q,p),τ: paths of (q, p) caused by a path of the tax known at date 0, with adjustment cost g."
    M = sequence_matrix(0.0, T_seq, g)              # only the supply and demand rows are used
    return np.linalg.solve(M[:2 * T_seq, :2 * T_seq], -M[:2 * T_seq, 2 * T_seq:])

Θ_qpτ = tax_effects()
T_e = Θ_qpτ.shape[1]
for name in ["demand", "supply"]:
    δ = δ_pop[name]
    implied = Θ_qpτ[:, :80] @ δ[:, 2]
    gap = max(np.abs(implied[:40] - δ[:40, 0]).max(),
              np.abs(implied[T_e:T_e + 40] - δ[:40, 1]).max())
    print(f"{name} shock: max |δ_(q,p) - Θ_(q,p),τ δ_τ| over 40 periods = {gap:.1e}")

Up to machine precision, the cross-regime differences in quantity and price are the effects of the cross-regime difference in the tax.

The next figure shows the treatments and their effects, each scaled so that the treatment is a unit subsidy at date 0.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
titles = ["treatment: tax $\\delta_\\tau$", "effect on quantity $\\delta_q$",
          "effect on price $\\delta_p$"]
for name, color in [("demand", 'C0'), ("supply", 'C1')]:
    δ = δ_pop[name] / -δ_pop[name][0, 2]
    for j, i in enumerate([2, 0, 1]):
        axes[j].plot(δ[:n_h, i], color=color, lw=2, label=f"{name} shock, population")
δ = δ_hat / -δ_hat[0, 2]
for j, i in enumerate([2, 0, 1]):
    axes[j].plot(δ[:, i], 'ko', ms=4, label="demand shock, interacted local projection")
    axes[j].axhline(0, color='k', lw=0.5)
    axes[j].set_title(titles[j])
    axes[j].set_xlabel("periods")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

The treatment identified by the demand shock is a subsidy at dates 0 and 1, while the demand shock keeps the price high, a small subsidy at date 2, and a smaller tax from date 3 on, once the price falls below normal.

The supply shock identifies a different treatment: a subsidy at date 0, almost nothing at date 1, and a tax from date 2 on.

Each non-policy shock identifies its own slice of $ \Theta_{(q,p),\tau} $.

Neither slice is a one-period change in the tax, which is what a contemporaneous shock to the rule delivers under permanent laissez-faire.

Manuel and Wolf [[2026](https://python-advanced.quantecon.org/zreferences.html#id352)] report the same contrast in U.S. data: interacting oil and fiscal shocks with the measure of the hawkishness of the Federal Open Market Committee constructed by Hack *et al.* [[2026](https://python-advanced.quantecon.org/zreferences.html#id368)] identifies gradual, forward-guidance-like paths for the policy rate, unlike the short-lived paths that conventional monetary policy shocks are known to identify [[Caravello *et al.*, 2026](https://python-advanced.quantecon.org/zreferences.html#id351)].

Exercise 31.4 uses these slices, in place of policy shocks, to construct McKay–Wolf counterfactuals.

Two shortcuts fall short.

First, the conditioning event must be a primitive shock, or a fixed combination of primitive shocks.

The innovation of a vector autoregression is not: in different regimes it combines current and past structural shocks with different weights, and the cross-regime comparison then mixes the effects of policy with the effects of a changing conditioning event (Exercise 31.5).

Second, comparing second moments across regimes, such as regime-specific autocovariances or the regime-dependent residual variances of [Vector autoregressions fit to regime-switching data](#pr-var), can reject the hypothesis that policy has no effects, as Mussa [[1986](https://python-advanced.quantecon.org/zreferences.html#id359)] argued for exchange-rate regimes.

Manuel and Wolf [[2026](https://python-advanced.quantecon.org/zreferences.html#id352)] show, however, that the set of policy effects consistent with how second moments change across regimes is generally far too wide to be informative.

## Summary

We constructed a Markov equilibrium model of a market for a good in which policy rules switch according to a Markov chain that agents know, and computed it by solving a Markov jump linear quadratic planning problem.

Decision rules depend on the current regime and on the probabilities of future regimes.

A switch within a known process leaves decision rules unchanged; a change in the process changes them in every regime.

With regimes as persistent as ours, the supply curve written in prices and quantities depends on the regime, contrary to the conjecture of Sims and Zha [[2006](https://python-advanced.quantecon.org/zreferences.html#id345)]: its coefficient on the price, relative to the tax, is about 60 percent larger under stabilization.

The regime enters the supply curve only through suppliers’ forecast of next period’s rule, so with regimes drawn afresh each period the supply curve would be the same in both regimes, as Sims and Zha assume.

A constant-coefficient vector autoregression for prices and quantities omits the tax, pools two policy rules, and has conditionally heteroskedastic innovations.

It fits almost as well as autoregressions that condition on the regime, but its impulse responses describe behavior in neither regime, although within a single regime the same autoregression approximately identifies the demand shock.

A reform that agents expect may be reversed has smaller effects in the first periods after a shock than the same reform made permanent and credible.

McKay and Wolf’s construction recovers outcomes under a permanent rule from data generated by switching regimes.

Applied on the tree of regime histories, as Caravello *et al.* [[2026](https://python-advanced.quantecon.org/zreferences.html#id351)] propose, it also recovers outcomes under a rule that may be reversed, path by path along every history and whatever rule generated the data; the probabilities of reversal are part of the specification of the counterfactual, and the change to the new process is still a surprise at date 0.

Differences across regimes in how a primitive non-policy shock propagates measure the effects of policy, as Manuel and Wolf [[2026](https://python-advanced.quantecon.org/zreferences.html#id352)] show, so regime switches can take the place of policy shocks; each non-policy shock identifies the effects of a different path of the tax.

The identification fails if the regime is correlated with other changes in how shocks propagate, and in samples of realistic length the identified effects are imprecise.

## Exercises

## Exercise 31.1

McKay and Wolf’s construction above uses expected responses conditional on the regime at the date of the shock.

Suppose instead that the econometrician averages the expected responses over the stationary distribution of the regime.

Show that the construction still recovers the response under a permanent stabilization rule, and explain why.

## Solution

The stationary distribution of the symmetric chain is $ (0.5, 0.5) $.

In [ ]:
def averaged(x0, weights):
    return sum(w * stacked(expected_responses(F, φs, Π, x0, s0, T_mw))
               for s0, w in enumerate(weights))

weights = [0.5, 0.5]
x_A = averaged(x0_demand, weights)
Θ = np.column_stack([averaged(np.eye(n)[i_ν + ℓ], weights) for ℓ in range(T_mw)])
R_new = np.hstack([np.zeros((T_mw, T_mw)), 2.0 * np.eye(T_mw), np.eye(T_mw)])
ν = np.linalg.solve(R_new @ Θ, -R_new @ x_A)
cf = (x_A + Θ @ ν).reshape(3, T_mw).T
print(f"max |McKay–Wolf - permanent stabilization| = {np.abs(cf[:30] - truth[:30]).max():.1e}")

Each conditional expected path satisfies the linear Euler equation and demand curve, and so does any weighted average of them.

The construction needs only that the baseline paths and the responses to policy shocks satisfy the private sector’s equations, and that the counterfactual rule is a linear restriction on paths.

## Exercise 31.2

Two regime processes can have the same stationary distribution and different persistence.

Compare the symmetric chain with switching probability 0.05 to a symmetric chain with switching probability 0.25.

Compute the expected response of quantity to a demand shock in each regime under each chain, and explain the differences.

## Solution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for pr, ls in [(0.05, '-'), (0.25, '--')]:
    Π_k = np.array([[1 - pr, pr], [pr, 1 - pr]])
    F_k = solve_regimes(φs, Π_k)
    for s0, ax in enumerate(axes):
        resp = expected_responses(F_k, φs, Π_k, x0_demand, s0, T_plot)
        ax.plot(resp[:, 0], lw=2, ls=ls, label=f"switching probability {pr}")
for ax, title in zip(axes, ["laissez-faire regime", "stabilization regime"]):
    ax.axhline(0, color='k', lw=0.5)
    ax.set_title(title)
    ax.set_xlabel("periods")
axes[0].legend()
plt.tight_layout()
plt.show()

With more frequent switches, agents in each regime expect the other regime sooner.

In the first four periods, responses in the laissez-faire regime rise toward those in the stabilization regime, and responses in the stabilization regime fall toward those in the laissez-faire regime.

From period 4 on, the stabilization-regime response is slightly higher under the less persistent chain, because a switch to laissez-faire, under which quantity adjusts more slowly, is more likely.

The expected paths also average over more switches along the path.

The stationary distribution alone does not determine behavior; persistence of the regimes matters.

## Exercise 31.3

In [Invertibility and Policy Counterfactuals](https://python-advanced.quantecon.org/policy_invertibility.html), a sufficiently aggressive price-stabilizing rule makes the supply shock fundamental for $ (q_t, p_t) $.

Repeat that calculation for the static-demand market of this lecture.

1. For permanent rules $ \phi \in \{0, 2, 4, 6, 8, 10\} $, compute the moving-average representation of $ (q_t, p_t) $ in terms of the two structural shocks, the innovation covariance $ V $ of its Wold representation, and $ \|L^{-1} d_s\| $, where $ L $ is the Cholesky factor of $ V $ and $ d_s $ is the impact response to the supply shock.  
1. For an economy that switches between $ \phi = 0 $ and $ \phi = 8 $, say in which regime a rotation of the innovations of a vector autoregression for $ (q_t, p_t) $ could recover the supply shock, and explain why the answer is approximate.  


*Hint:* write the moving-average representation, truncated after 80 lags, in state-space form and solve the Kalman filter’s Riccati equation with `scipy.linalg.solve_discrete_are`.

## Solution

The structural moving-average coefficients are the expected responses to the two shocks.

The steady-state Kalman filter for the truncated moving average delivers the innovation covariance $ V $, as in [Dynamic Supply and Demand Curves](https://python-advanced.quantecon.org/supply_demand_var.html).

In [ ]:
from scipy.linalg import solve_discrete_are

H_ma = 80

def ma_coefficients(φ):
    "D_j (2 x 2): responses of (q, p) at lag j to the supply and demand shocks under a permanent rule φ."
    F_φ = solve_regimes([φ], one)
    D = np.zeros((H_ma, 2, 2))
    for k, x0 in enumerate([x0_supply, x0_demand]):
        D[:, :, k] = expected_responses(F_φ, [φ], one, x0, 0, H_ma)[:, :2]
    return D

def innovation_covariance(D):
    "Innovation covariance of the Wold representation of z_t = Σ_j D_j ε_(t-j)."
    k = D.shape[2]
    A_ma = np.zeros((H_ma * k, H_ma * k)); A_ma[k:, :-k] = np.eye((H_ma - 1) * k)
    C_ma = np.zeros((H_ma * k, k)); C_ma[:k] = np.eye(k)
    G = np.hstack(list(D))
    R_meas = 1e-10 * np.eye(2)
    Σ = solve_discrete_are(A_ma.T, G.T, C_ma @ C_ma.T, R_meas, balanced=False)
    return G @ Σ @ G.T + R_meas

print(" φ    ||L^-1 d_s||   ||L^-1 d_d||")
for φ in [0.0, 2.0, 4.0, 6.0, 6.6, 6.7, 8.0, 10.0]:
    D = ma_coefficients(φ)
    L = np.linalg.cholesky(innovation_covariance(D))
    norms = np.linalg.norm(np.linalg.solve(L, D[0]), axis=0)
    print(f"{φ:4.1f}   {norms[0]:12.4f}   {norms[1]:12.4f}")

The demand shock is fundamental under every rule.

The supply shock is not fundamental under laissez-faire; it becomes fundamental at about $ \phi = 6.7 $, a more aggressive rule than the market with dynamic demand of [Invertibility and Policy Counterfactuals](https://python-advanced.quantecon.org/policy_invertibility.html) requires.

An economy that switches between $ \phi = 0 $ and $ \phi = 8 $ alternates between a regime in which a vector autoregression for $ (q_t, p_t) $ cannot recover the supply shock and a regime in which a rotation of its innovations can.

The statement is approximate for two reasons.

Regime-conditional decision rules differ from those under permanent rules, a difference that vanishes as the regimes become permanent.

Fundamentalness is a property of a time-invariant moving average, while the innovations of an autoregression fit to switching data project on a past that includes periods spent in the other regime.

A constant-coefficient autoregression fit to the pooled data mixes the two regimes.

## Exercise 31.4

Suppose that the econometrician has no evidence on policy shocks.

She has only the two treatments identified in [Learning the effects of policy from regime switches](#pr-ilp) by interacting the demand and supply shocks with the regime, together with their effects on quantity and price.

1. Starting from the expected responses to a demand shock in the laissez-faire regime, use the least-squares version of McKay and Wolf’s construction from [Policy Counterfactuals from Impulse Responses](https://python-advanced.quantecon.org/policy_counterfactuals.html) to approximate the response to a demand shock under permanent stabilization, $ \tau_t = -2 p_t $, using (a) the treatment identified by the demand shock, (b) the treatment identified by the supply shock, and © both.  
1. Repeat for the response to a supply shock.  
1. Explain the pattern of errors.  

## Solution

Each treatment, stacked as a path of $ (q, p, \tau) $, plays the role of a column of $ \Theta_\nu $.

In [ ]:
slices = {name: stacked(δ_pop[name][:T_mw]) for name in ["demand", "supply"]}
R_new = np.hstack([np.zeros((T_mw, T_mw)), 2.0 * np.eye(T_mw), np.eye(T_mw)])

print("shock    treatments used     weights          max error in q / peak")
for name, x0 in [("demand", x0_demand), ("supply", x0_supply)]:
    x_A_sh = stacked(expected_responses(F, φs, Π, x0, 0, T_mw))
    truth_q = expected_responses(F_st, [2.0], one, x0, 0, T_mw)[:, 0]
    for used in [["demand"], ["supply"], ["demand", "supply"]]:
        Ω = np.column_stack([slices[u] for u in used])
        weights, *_ = np.linalg.lstsq(R_new @ Ω, -R_new @ x_A_sh, rcond=None)
        q_cf = (x_A_sh + Ω @ weights)[:T_mw]
        err = np.abs(q_cf[:30] - truth_q[:30]).max() / np.abs(truth_q).max()
        w_str = ", ".join(f"{w:.3f}" for w in weights)
        print(f"{name:7s}  {' + '.join(used):18s}  {w_str:15s}  {err:8.3f}")
    gap = np.abs(x_A_sh[:30] - truth_q[:30]).max() / np.abs(truth_q).max()
    print(f"{name:7s}  (no adjustment)                       {gap:8.3f}")

For each shock, the treatment identified by that shock delivers an accurate counterfactual, with a weight close to one.

The treatment identified by the other shock does not help: for the demand shock it leaves the error almost unchanged, and for the supply shock it makes the error larger than no adjustment at all, 0.174 against 0.135.

The reason is that the counterfactual rule is one of the two historical regimes.

The response to a demand shock in the stabilization regime differs from the response under *permanent* stabilization only because of switches: agents anticipate them, and the expected path averages over those that occur.

So the laissez-faire response plus the cross-regime difference, which is the stabilization-regime response, is already close to the counterfactual, and the remaining error reflects switches.

The weights are not comparable across slices, because the slices differ in scale: the supply shock’s cross-regime difference in the date-0 tax is 0.06, about 48 times smaller than the demand shock’s, which is why its weight is large when it is used for the demand shock.

For a counterfactual rule outside the range of the historical regimes, or for a shock whose own cross-regime comparison is unavailable, the slices identified by other shocks would have to do the work, and the least-squares approximation could be poor; Exercise 31.10 explores the first case.

## Exercise 31.5

Replace the demand shock in the interacted local projections [(31.4)](#equation-pr-ilp) by the innovation in the price from an autoregression of order 8 fit to simulated $ (q_t, p_t) $, orthogonalized with the price ordered first.

Check whether the estimated effect on quantity equals $ \Theta_{(q,p),\tau} $ times the estimated treatment, and compare with the projections on the demand shock.

## Solution

In [ ]:
lags = 8
coef, U = var_fit(y_sim[:, :2], lags)
swap = np.array([[0, 1], [1, 0]])
L_p = swap @ np.linalg.cholesky(swap @ np.cov(U.T) @ swap) @ swap
price_innovation = np.concatenate([np.zeros(lags), np.linalg.solve(L_p, U.T)[1]])

for label, shock in [("demand shock", -w_sim[:, 1]), ("VAR price innovation", price_innovation)]:
    δ_est = interacted_lp(y_sim, shock, ζ, n_h)
    q_implied = Θ_qpτ[:n_h, :n_h] @ δ_est[:, 2]
    gap = np.abs(δ_est[:, 0] - q_implied).max() / np.abs(δ_est[:, 0]).max()
    print(f"{label:21s}: max |δ_q - Θ_q,τ δ_τ| relative to the peak of δ_q = {gap:.3f}")

With the demand shock, the two sides differ only by sampling error.

With the price innovation, they differ by much more.

The price innovation is a combination of current and past structural shocks, and both the combination and its propagation depend on the regime path.

The cross-regime comparison then mixes the effects of policy with the effects of a conditioning event that itself changes with the regime.

(Truncating $ \Theta_{q,\tau} $ at 12 horizons, as the code does, ignores the tail of the treatment; the demand-shock row shows that this matters little.)

## Exercise 31.6

In [Two levels of the Lucas critique](#pr-levels), a reform to the stabilization rule arrives with probability $ r $ each period and is permanent.

Use the tree construction of [A counterfactual rule that may be reversed](#pr-tree), with the objects `x_A_tree` and `Θ_tree` taken from the switching economy, to compute the expected response to a demand shock that occurs under laissez-faire before any reform, for $ r = 0.05 $ and $ r = 0.2 $.

Compare the results with the Markov jump linear quadratic solution.

## Solution

The tree is the same as before with the two rules interchanged: laissez-faire at date 0 and a permanent switch to stabilization with probability $ r $ each period.

In [ ]:
for r in [0.05, 0.2]:
    Π_r = np.array([[1 - r, r], [0.0, 1.0]])
    truth_r = expected_responses(solve_regimes(φs, Π_r), φs, Π_r, x0_demand, 0, 30)
    tree_r, _, _ = mw_tree(x_A_tree, Θ_tree, 0.0, 2.0, r)
    print(f"r = {r}: impact on q, tree {tree_r[0, 0]:.4f}, Markov jump LQ {truth_r[0, 0]:.4f}; "
          f"max gap over 30 periods {np.abs(tree_r[:30] - truth_r).max():.1e}")

The tree construction reproduces the effect of a reform that may arrive, including the larger impact response of quantity under laissez-faire that the prospect of the reform produces.

The news that agents receive at each date is whether the reform has arrived.

## Exercise 31.7

Manuel and Wolf [[2026](https://python-advanced.quantecon.org/zreferences.html#id352)] warn that interacted local projections fail when the regime indicator is correlated with some other source of change in how shocks propagate.

Suppose that the switch to stabilization coincides with a change in the private sector: suppliers’ adjustment cost is $ g_s = 10 $ under laissez-faire and $ g_s = 5 $ under stabilization, as if the reform came with a deregulation of supply.

1. Compute the decision rules of the switching economy, now with a regime-specific adjustment cost.  
1. Compute the population interacted local projections $ \delta $ for the demand shock.  
1. Check whether $ (\delta_q, \delta_p) = \Theta_{(q,p),\tau}\, \delta_\tau $, using the private sector’s equations with $ g_s = 10 $ and with $ g_s = 5 $.  

## Solution

With a regime-specific adjustment cost, the supplier’s Euler equation becomes $ h_s q_t + g_{s^p_t}(q_t - q_{t-1}) - \beta\, \mathbb{E}_t\, g_{s^p_{t+1}}(q_{t+1} - q_t) + \ldots = 0 $.

It is still the first-order condition of the planning problem, now with an adjustment-cost weight that depends on the regime, so the matrices $ R_i $, $ Q_i $, and $ N_i $ all become regime specific.

In [ ]:
F_g = solve_regimes(φs, Π, gs=[10.0, 5.0])
δ_g = (expected_responses(F_g, φs, Π, x0_demand, 1, 80)
       - expected_responses(F_g, φs, Π, x0_demand, 0, 80))

for g in [10.0, 5.0]:
    Θ_g = tax_effects(g=g)
    implied = Θ_g[:, :80] @ δ_g[:, 2]
    gap = np.abs(implied[:40] - δ_g[:40, 0]).max() / np.abs(δ_g[:40, 0]).max()
    print(f"private sector with g_s = {g:4.1f}: max |δ_q - Θ_q,τ δ_τ| relative to the peak of δ_q = {gap:.2f}")

Neither version of the private sector’s equations maps the cross-regime difference in the tax into the cross-regime difference in quantity.

The difference across regimes now combines the effect of the tax with the effect of the change in adjustment costs, and the projections attribute both to policy.

This is the threat to identification that Manuel and Wolf [[2026](https://python-advanced.quantecon.org/zreferences.html#id352)] emphasize: in applications, the analyst must argue that the measure of policy is unrelated to other sources of state dependence.

## Exercise 31.8

The regime indicator need not be the regime itself.

Manuel and Wolf [[2026](https://python-advanced.quantecon.org/zreferences.html#id352)] use a measure of the hawkishness of the Federal Open Market Committee constructed by Hack *et al.* [[2026](https://python-advanced.quantecon.org/zreferences.html#id368)], a variable that is correlated with policy without being policy.

Rerun the interacted local projections of [Learning the effects of policy from regime switches](#pr-ilp) on the simulated data with two imperfect measures of the regime:

1. last period’s regime, and  
1. the current regime misclassified with probability 0.2.  


Check whether the causal restriction [(31.5)](#equation-pr-ilp-effects) still holds, and compare the estimated
treatment with the one that the true regime identifies.

## Solution

In [ ]:
rng = np.random.default_rng(0)
ζ_lag = np.concatenate([[0.0], ζ[:-1]])
ζ_noisy = np.where(rng.random(len(ζ)) < 0.2, 1 - ζ, ζ)

for label, z in [("regime", ζ), ("last period's regime", ζ_lag), ("misclassified regime", ζ_noisy)]:
    δ_z = interacted_lp(y_sim, -w_sim[:, 1], z, n_h)
    q_implied = Θ_qpτ[:n_h, :n_h] @ δ_z[:, 2]
    gap = np.abs(δ_z[:, 0] - q_implied).max() / np.abs(δ_z[:, 0]).max()
    print(f"{label:22s}: causal-restriction gap {gap:.3f}, "
          f"treatment at date 0 relative to the true regime's {δ_z[0, 2] / δ_hat[0, 2]:.2f}")

Both imperfect measures satisfy the causal restriction up to sampling error: each moves the tax and nothing else, so each identifies a valid treatment.

The treatments are weaker.

Last period’s regime predicts this period’s regime with probability 0.95, so the treatment is $ \Pi_{11} - \Pi_{01} = 0.9 $ times the one that the true regime identifies; misclassification with probability 0.2 scales it by $ 1 - 2 \times 0.2 = 0.6 $.

With two regimes that follow an exogenous Markov chain, as here, every valid measure identifies a scaled version of this treatment.

Because the chain is exogenous and Markov, the response to the shock depends on the past only through the current regime, so the expected response conditional on a measure $ \zeta_t $ is a weighted average of the two regime responses, and the projections deliver $ \Pr(s^p_t = 1 \mid \zeta_t = 1) - \Pr(s^p_t = 1 \mid \zeta_t = 0) $ times the cross-regime difference.

Different measures can identify different treatments with more than two regimes, or when switching depends on the state of the economy, as in the endogenous rules that Manuel and Wolf [[2026](https://python-advanced.quantecon.org/zreferences.html#id352)] allow.

## Exercise 31.9

The tree of [A counterfactual rule that may be reversed](#pr-tree) has one branch for each reversal date because a reversal is permanent.

Return to a recurrent chain, $ \Pi = \begin{pmatrix} 0.95 & 0.05 \\ e & 1 - e \end{pmatrix} $ with $ e = 0.2 $, under which laissez-faire can give way to stabilization again, starting in stabilization.

Its tree doubles in size every period, so truncate it: regimes can switch through date $ K $, and the regime in place at date $ K $ is permanent.

1. Write the tree construction for a general chain with this truncation, and check it against a Markov jump linear quadratic solution of the truncated process, obtained by iterating the linked Riccati equations backward from the permanent-regime values at date $ K $.  
1. For $ K = 1, \ldots, 6 $, report the number of unknowns and how far the truncated counterfactual lies from the expected response under the untruncated chain.  

## Solution

The truncated process has time-varying decision rules.

From date $ K $ on, each regime is permanent, so $ P_{i,K} $ is the value under a permanent rule; for $ t < K $, the linked Riccati equations run backward with $ \bar P_{i,t} = \sum_j \Pi_{ij} P_{j,t+1} $.

In [ ]:
def riccati_step(P_next, Π_i, mats_i):
    "One step of the linked Riccati equations: (P_i, F_i) given next period's values."
    Q, R, N = mats_i
    P_bar = np.tensordot(Π_i, P_next, axes=1)
    G = β * B.T @ P_bar @ A + N
    F_i = np.linalg.solve(Q + β * B.T @ P_bar @ B, G)
    return R + β * A.T @ P_bar @ A - G.T @ F_i, F_i

def truncated_truth(φs, Π, K, s0, x0, T_out):
    "Expected responses when the regime switches through date K and is permanent afterward."
    m = len(φs)
    mats = [loss_matrices(φ) for φ in φs]
    F_perm, P = solve_regimes(φs, np.eye(m), return_P=True)    # values with permanent regimes
    F_t = [None] * K
    for t in range(K - 1, -1, -1):              # backward from date K
        steps = [riccati_step(P, Π[i], mats[i]) for i in range(m)]
        P = np.array([s[0] for s in steps])
        F_t[t] = np.array([s[1] for s in steps])
    M = [x0 * (j == s0) for j in range(m)]
    out = np.zeros((T_out, 3))
    for t in range(T_out):
        F_now, Π_now = (F_t[t], Π) if t < K else (F_perm, np.eye(m))
        q_j = [-(F_now[j] @ M[j]).item() for j in range(m)]
        p_j = [-d_sel @ M[j] - h_d * q_j[j] for j in range(m)]
        out[t] = sum(q_j), sum(p_j), sum(-φs[j] * p_j[j] for j in range(m))
        M = [sum(Π_now[j, k] * (A - B @ F_now[j]) @ M[j] for j in range(m)) for k in range(m)]
    return out

On the tree, a node is a history of regimes $ (s^p_0, \ldots, s^p_t) $ with $ t \le K $.

The news that arrives at the children of a node averages to zero, so the news at one child is a linear combination of the news at the others; we keep the news at all but the last child as unknowns.

In [ ]:
def mw_tree_general(x_A, Θ, φs, Π, K, s0, T=T_tree):
    """
    McKay–Wolf counterfactual on the tree of regime histories through date K,
    with the regime in place at date K permanent.  Returns expected paths and the
    number of unknowns.
    """
    m = len(φs)
    histories = [[(s0,)]]
    for t in range(1, K + 1):
        histories.append([h + (j,) for h in histories[-1] for j in range(m) if Π[h[-1], j] > 0])
    news = {(s0,): [(0, 1.0)]}                   # node -> list of (start of an unknown block, weight)
    n_unknowns = T
    for t in range(1, K + 1):
        for h in histories[t - 1]:
            kids = [h + (j,) for j in range(m) if Π[h[-1], j] > 0]
            prob = [Π[h[-1], c[-1]] for c in kids]
            blocks = []
            for c, pr in zip(kids[:-1], prob[:-1]):
                news[c] = [(n_unknowns, 1.0)]
                blocks.append((n_unknowns, pr))
                n_unknowns += T - t
            news[kids[-1]] = [(b, -pr / prob[-1]) for b, pr in blocks]

    Θ_rule = [Θ[2 * T:] + φ * Θ[T:2 * T] for φ in φs]
    rows, rhs = [], []
    def rule_row(h, t):
        "The rule τ_t + φ p_t = 0 at date t along the history h."
        row, j = np.zeros(n_unknowns), h[min(t, K)]
        for s in range(min(t, K) + 1):
            for b, wgt in news[h[:s + 1]]:
                row[b:b + T - s] += wgt * Θ_rule[j][t - s, :T - s]
        rows.append(row)
        rhs.append(-(x_A[2 * T + t] + φs[j] * x_A[T + t]))
    for t in range(K + 1):
        for h in histories[t]:
            rule_row(h, t)
    for h in histories[K]:
        for t in range(K + 1, T):
            rule_row(h, t)
    sol = np.linalg.solve(np.array(rows), np.array(rhs))

    expected = np.zeros((T, 3))
    for h in histories[K]:
        path = x_A.reshape(3, T).T.copy()
        for s in range(K + 1):
            arriving = np.zeros(T - s)
            for b, wgt in news[h[:s + 1]]:
                arriving += wgt * sol[b:b + T - s]
            for blk in range(3):
                path[s:, blk] += Θ[blk * T:(blk + 1) * T - s, :T - s] @ arriving
        expected += np.prod([Π[h[i], h[i + 1]] for i in range(K)]) * path
    return expected, n_unknowns

x_A_seq, Θ_seq = sequence_space_objects(0.5)
e = 0.2
Π_rec = np.array([[0.95, 0.05], [e, 1 - e]])
untruncated = expected_responses(solve_regimes(φs, Π_rec), φs, Π_rec, x0_demand, 1, 30)

peak = np.abs(untruncated[:, 0]).max()
print(" K   unknowns   |tree - truncated LQ|   |truncated - untruncated| in q, percent of peak")
for K in range(1, 7):
    tree_K, size = mw_tree_general(x_A_seq, Θ_seq, φs, Π_rec, K, 1)
    truth_K = truncated_truth(φs, Π_rec, K, 1, x0_demand, 30)
    gap_K = np.abs(tree_K[:30, 0] - untruncated[:, 0]).max()
    print(f"{K:2d}   {size:8d}   {np.abs(tree_K[:30] - truth_K).max():21.1e}   "
          f"{100 * gap_K / peak:40.3f}")

The tree reproduces the truncated process to machine precision for every $ K $.

The number of unknowns roughly doubles with each additional date of switching, while the gap to the untruncated chain shrinks by a factor that rises from about 1.7 to about 5 per date: switches far in the future matter little for responses that die out quickly.

By $ K = 6 $, the truncated counterfactual lies within about 0.013 percent of the peak response of quantity under the untruncated chain.

With an absorbing laissez-faire regime, the same code adds only one history per date; with $ K = 59 $ it reproduces the calculation of [A counterfactual rule that may be reversed](#pr-tree), whose tree runs for 60 dates.

## Exercise 31.10

The treatments identified in [Learning the effects of policy from regime switches](#pr-ilp) come from two historical regimes, $ \phi = 0 $ and $ \phi = 2 $.

Repeat Exercise 31.4 for two counterfactual rules that are not historical regimes: $ \tilde \phi = 1 $, between the two, and $ \tilde \phi = 5 $, outside their range.

For each rule and each shock, compare the error of the least-squares counterfactual that uses the shock’s own treatment with the error that uses both treatments.

## Solution

In [ ]:
# uses the treatments `slices` from the solution to pr_ex4
print("rule   shock    no adjustment   own treatment   both treatments")
for φ_t in [1.0, 5.0]:
    R_t = np.hstack([np.zeros((T_mw, T_mw)), φ_t * np.eye(T_mw), np.eye(T_mw)])
    F_t = solve_regimes([φ_t], one)
    for name, x0 in [("demand", x0_demand), ("supply", x0_supply)]:
        x_A_sh = stacked(expected_responses(F, φs, Π, x0, 0, T_mw))
        truth_q = expected_responses(F_t, [φ_t], one, x0, 0, T_mw)[:, 0]
        errors = [np.abs(x_A_sh[:30] - truth_q[:30]).max() / np.abs(truth_q).max()]
        for used in [[name], ["demand", "supply"]]:
            Ω = np.column_stack([slices[u] for u in used])
            weights, *_ = np.linalg.lstsq(R_t @ Ω, -R_t @ x_A_sh, rcond=None)
            q_cf = (x_A_sh + Ω @ weights)[:T_mw]
            errors.append(np.abs(q_cf[:30] - truth_q[:30]).max() / np.abs(truth_q).max())
        print(f"{φ_t:4.1f}   {name:7s}  {errors[0]:13.3f}   {errors[1]:13.3f}   {errors[2]:15.3f}")

Between the historical regimes, the treatments identified by regime switches remain informative: with $ \tilde \phi = 1 $, the shock’s own treatment brings the error to 1.8 percent for the demand shock and 1.0 percent for the supply shock, and both treatments together do a little better.

Outside the historical range, the approximation deteriorates: with $ \tilde \phi = 5 $, the demand shock’s own treatment leaves an error of 8.3 percent, which falls to 4.3 percent when both treatments are used, while the error for the supply shock stays below 2 percent.

The cross-regime comparisons identify the effects of tax paths like the ones that the historical regimes produced; a rule much more aggressive than either calls for larger and differently shaped tax paths, which the two slices span only approximately.

That is the limited-evidence problem of [Policy Counterfactuals from Impulse Responses](https://python-advanced.quantecon.org/policy_counterfactuals.html) again, and the remedy of [When do we need a model?](https://python-advanced.quantecon.org/policy_counterfactuals.html#pc-hybrid), a model fitted to the evidence, applies here too.